### Year 5 Topology 6 Contingency Analysis Reporting - Lower Limit Violation

In [1]:
import pandas as pd
import numpy as np

#### Bus Voltage Data

In [2]:
list_volt = []
list_gens = [0, 50, 100]
list_lsc = ['LLS','RLS','HLS']
list_gen_hydro = [400,500,600] #dry, average, wet hydological scenarios
list_gen_wind = [0,40,75] # for 3 wind gust scenarios
for gen in list_gens:
    for gen_hy in list_gen_hydro:
        for lsc in list_lsc:
            for gen_wi in list_gen_wind:
                file_out = 'savnw_sol_' + str(gen) +'_hy_' +str(gen_hy) +'_wi_' + str(gen_wi) +'_' + str(lsc)+'.xlsx'
                data_volt = pd.read_excel(file_out, sheet_name='Bus Voltage', usecols = ['BUS', 'RECORD', 'TYPE', 'MIN/DROP', 'MAX/RISE', 'CONTINGENCY',
                                        'BASE VOLTS', 'CONT VOLTS', 'DEVIATION', 'RANGE VIO', 'DEV VIO'] )
                data_vo = data_volt.dropna(how='all').reset_index(drop=True)
                data_vo['Scenario']= 'Solar = ' + str(gen) + ' MW, ' + 'Hydro = ' + str(gen_hy) + ' MW, ' + 'Wind = ' + str(gen_wi) + ' MW, '  + lsc.upper()
                list_volt.append(data_vo)
data_v = pd.concat(list_volt).reset_index(drop=True)

#### Bus voltage data wrangling 

In [3]:
data_v = data_v.rename(columns={'BUS':'Bus', 
                                'CONTINGENCY':'Contingency',
                                'BASE VOLTS':'Base Voltage',                               
                                'CONT VOLTS':'Contingency Voltage',
                                'RANGE VIO' : 'Range Violation', 
                                'DEVIATION' : 'Deviation'})
data_v['Contingency'] = data_v['Contingency'].str.replace('&','\&')
data_v['Bus'] = data_v['Bus'].str.replace('_','\_')
data_v['Bus Number'] = data_v['Bus'].str.split(expand=True)[0]

#### Lower limit bus voltage violations 

In [4]:
data_low = data_v[data_v['Contingency Voltage']<0.9].reset_index(drop=True)
print(len(data_low))
display(data_low.head())

3118


,Bus,RECORD,TYPE,MIN/DROP,MAX/RISE,Contingency,Base Voltage,Contingency Voltage,Deviation,Range Violation,DEV VIO,Scenario,Bus Number
0,154 DOWNTN 230.00,ALL,RANGE,0.95,1.05,SING OPN LIN 10 201-202(1),0.971035,0.891920,-0.079114,-0.058080,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS",154
1,154 DOWNTN 230.00,LIMIT,RANGE,0.90,1.10,SING OPN LIN 10 201-202(1),0.971035,0.891920,-0.079114,-0.008080,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS",154
2,103 SOLAR\_PV 13.800,ALL,RANGE,0.95,1.05,BUS 205,0.999110,0.794608,-0.204502,-0.155392,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS",103
3,152 MID500 500.00,ALL,RANGE,0.95,1.05,BUS 205,1.025738,0.888031,-0.137707,-0.061969,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS",152
4,153 MID230 230.00,ALL,RANGE,0.95,1.05,BUS 205,1.004358,0.850264,-0.154094,-0.099735,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS",153


#### Reporting of Buses with lower voltage violations

In [5]:
L_scen = list(data_low['Scenario'].unique())
print(f'It was seen that for Topology 4, all the studied scenarios reported lower voltage limit violation.')
print('------------------------------------------------------------------------------------------------------------------------------------------------------')
for vlscen in L_scen:
    print(vlscen)
    # Extracting bus data with voltage less than 0.9 PU
    data_lscen = data_low[data_low['Scenario'] == vlscen]
    print(f'For the studied scenario {vlscen}, the bus(es) reporting lower emergency range violation(s) are the bus(es)',', '.join(data_lscen['Bus Number'].unique()),)
    # lower range violation latex table data reporting
    data_replv = data_lscen[['Bus Number','Contingency','Base Voltage', 'Contingency Voltage', 'Deviation', 'Range Violation']].reset_index(drop=True)
    
    unit_lowv = data_replv[data_replv['Contingency'].str.contains('UNIT')]
    if len(unit_lowv) != 0:
        unit_lv = unit_lowv.drop_duplicates(subset=['Bus Number'])
        for contlu in list(unit_lv['Contingency'].unique()):
            unit_cont = unit_lv[unit_lv['Contingency']==contlu]
            busesu = ', '.join(unit_cont['Bus Number'].unique())
            print(f'The bus(es) {busesu} reported violations for unit fault {contlu}')
       
    bus_lowv = data_replv[data_replv['Contingency'].str.contains('BUS')]
    if len(bus_lowv) != 0:
        bus_lv = bus_lowv.drop_duplicates(subset=['Bus Number'])
        for contlb in list(bus_lv['Contingency'].unique()):
            bus_cont = bus_lv[bus_lv['Contingency']==contlb]
            busesb = ', '.join(bus_cont['Bus Number'].unique())
            print(f'The bus(es) {busesb} reported violations for the bus fault {contlb}')
                    
    line_lowv = data_replv[data_replv['Contingency'].str.contains('SING OPN LIN')]
    if len(line_lowv) != 0:
        line_lv = line_lowv.drop_duplicates(subset=['Bus Number'])
        for contll in list(line_lv['Contingency'].unique()):
            line_cont = line_lv[line_lv['Contingency']==contll]
            busesl = ', '.join(line_cont['Bus Number'].unique())
            print(f'The bus(es) {busesl} reported violations for the single line open fault {contll}')
    print('------------------------------------------------------------------------------------------------------------------------------------------------------')


It was seen that for Topology 4, all the studied scenarios reported lower voltage limit violation.
------------------------------------------------------------------------------------------------------------------------------------------------------
Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS
For the studied scenario Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS, the bus(es) reporting lower emergency range violation(s) are the bus(es) 154, 103, 152, 153, 202, 203, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3012, 3018
The bus(es) 103, 152, 153, 154, 202, 203, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3012, 3018 reported violations for the bus fault BUS 205
The bus(es) 154 reported violations for the single line open fault SING OPN LIN   10 201-202(1)
------------------------------------------------------------------------------------------------------------------------------------------------------
Solar = 0 MW, Hydro = 400 MW, Wind = 40 MW, LLS
For the studied scenario Solar = 0 MW,

Lower Voltage Limit Counts - Scenario 

In [6]:
vl_index = list(data_low['Scenario'].value_counts().index)
vl_counts = list(data_low['Scenario'].value_counts())
dict_vl_count = {
    'Scenario':vl_index,
    'Violation Counts':vl_counts
}
scen_lv_vc  = pd.DataFrame(dict_vl_count)
scen_lv_vc

,Scenario,Violation Counts
0,"Solar = 50 MW, Hydro = 600 MW, Wind = 75 MW, HLS",92
1,"Solar = 50 MW, Hydro = 600 MW, Wind = 0 MW, HLS",90
2,"Solar = 50 MW, Hydro = 600 MW, Wind = 40 MW, HLS",86
3,"Solar = 50 MW, Hydro = 400 MW, Wind = 40 MW, RLS",66
4,"Solar = 50 MW, Hydro = 400 MW, Wind = 0 MW, RLS",66
...,...,...
76,"Solar = 0 MW, Hydro = 600 MW, Wind = 75 MW, RLS",10
77,"Solar = 100 MW, Hydro = 500 MW, Wind = 40 MW, LLS",8
78,"Solar = 100 MW, Hydro = 500 MW, Wind = 0 MW, LLS",8
79,"Solar = 100 MW, Hydro = 500 MW, Wind = 75 MW, LLS",8


Lower Voltage Limit Counts - Bus

In [7]:
vlb_index = list([i.strip().split()[0] for i in data_low['Bus'].value_counts().index])
vlb_counts = list(data_low['Bus'].value_counts())
dict_vlb_count = {
    'Bus':vlb_index,
    'Violation Counts':vlb_counts
}
bus_lv_vc  = pd.DataFrame(dict_vlb_count)
bus_lv_vc

,Bus,Violation Counts
0,154,944
1,205,614
2,3008,352
3,103,244
4,3007,186
5,204,182
6,153,104
7,3012,94
8,3006,92
9,3009,78


Lower Voltage Limit Counts - Contingency

In [8]:
vlc_index = list(data_low['Contingency'].value_counts().index)
vlc_counts = list(data_low['Contingency'].value_counts())
dict_vlc_count = {
    'Contingency':vlc_index,
    'Violation Counts':vlc_counts
}
cont_lv_vc  = pd.DataFrame(dict_vlc_count)
cont_lv_vc

,Contingency,Violation Counts
0,SING OPN LIN 6 152-153(1),576
1,BUS 203,364
2,BUS 3005,312
3,SING OPN LIN 10 201-202(1),306
4,BUS 153,242
5,BUS 205,198
6,BUS 3003,158
7,BUS 3004,158
8,UNIT 3018(1),66
9,SING OPN LIN 35 3008-3018(1),54


Lower Voltage Result Summary - grouped by Scenario and Contingency 

In [9]:
data_fil_lv = data_low[['Scenario', 'Bus Number', 'Contingency']].drop_duplicates()
pivot_lv = pd.DataFrame(pd.pivot(data_fil_lv, index= ['Scenario','Contingency'], columns = 'Bus Number',values = 'Bus Number').to_records())
pivot_lv['Buses'] = pivot_lv['103'].astype(str).str.cat(pivot_lv[['152', '153', '154', '202', '203',
       '204', '205', '3004', '3005', '3006', '3007', '3008', '3009', '3010',
       '3012', '3018']].astype(str), sep=',')
pivot_lv['Buses'] =pivot_lv['Buses'].str.replace(',nan','').str.replace('nan,','')
pivot_low = pivot_lv.drop(columns = list(data_fil_lv['Bus Number'].unique()))

In [10]:
pivot_table_lv = pd.DataFrame(pd.pivot_table(data_fil_lv, index= ['Scenario','Contingency'],values = 'Bus Number', aggfunc='count').to_records())
pivot_table_low = pivot_table_lv.rename(columns = {'Bus Number':'Bus Count'})
df_low = pd.merge(pivot_low, pivot_table_low, on=['Scenario','Contingency'], how='inner').sort_values(by='Bus Count', ascending=False).reset_index(drop=True)
df_low.head(60)

,Scenario,Contingency,Buses,Bus Count
0,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, LLS",BUS 205,"103,152,153,154,202,203,3004,3005,3006,3007,30...",15
1,"Solar = 0 MW, Hydro = 400 MW, Wind = 40 MW, LLS",BUS 205,"103,152,153,154,202,203,3004,3005,3006,3007,30...",15
2,"Solar = 50 MW, Hydro = 400 MW, Wind = 0 MW, RLS",SING OPN LIN 10 201-202(1),"103,152,153,154,202,203,204,205,3006,3007,3008...",14
3,"Solar = 50 MW, Hydro = 400 MW, Wind = 40 MW, RLS",SING OPN LIN 10 201-202(1),"103,152,153,154,202,203,204,205,3006,3007,3008...",14
4,"Solar = 50 MW, Hydro = 400 MW, Wind = 75 MW, RLS",SING OPN LIN 10 201-202(1),"103,152,153,154,202,203,204,205,3006,3007,3008...",14
5,"Solar = 50 MW, Hydro = 400 MW, Wind = 75 MW, LLS",BUS 205,"103,152,153,154,203,3005,3006,3007,3008,3009,3...",13
6,"Solar = 100 MW, Hydro = 400 MW, Wind = 0 MW, RLS",SING OPN LIN 10 201-202(1),"103,153,154,202,203,204,205,3006,3007,3008,300...",13
7,"Solar = 100 MW, Hydro = 600 MW, Wind = 0 MW, LLS",SING OPN LIN 10 201-202(1),"103,153,154,202,203,204,205,3006,3007,3008,300...",13
8,"Solar = 50 MW, Hydro = 400 MW, Wind = 40 MW, HLS",SING OPN LIN 6 152-153(1),"103,153,154,203,204,205,3006,3007,3008,3009,30...",12
9,"Solar = 50 MW, Hydro = 400 MW, Wind = 75 MW, HLS",SING OPN LIN 6 152-153(1),"103,153,154,203,204,205,3006,3007,3008,3009,30...",12
